### Installling Ultralytics

In [ ]:
# install the required folder
!pip install ultralytics

### Connect to Google drive

In [1]:
from google.colab import drive
from pathlib import Path
import torch
drive.mount('/content/drive')


Mounted at /content/drive


### Specify the Image Path for the Project

The path on drive is structured like this. Consult this page to see how to structure the folder in each round. (https://docs.ultralytics.com/datasets/classify#dataset-structure-for-yolo-classification-tasks)
```
AI_bile_training/
├── round_1/
│   ├── train/
│   └── val/
├── round_2/
├── round_3/
├── round_train/
└── result/
    ├── train/
    ├── round_1/
    ├── round_2/
    └── round_3/
```



In [ ]:

# the folder storing the bile image data
image_path = "/content/drive/MyDrive/AI_bile_training"

DATASET_ROOT = Path(r"/content/drive/MyDrive/AI_bile_training")
RESULT_ROOT = Path(r"/content/drive/MyDrive/AI_bile_training/result")

DATASET_PATH = Path(r"/content/drive/MyDrive/AI_bile_training")

FOLDS = [
    # "fold_0",
    # "fold_1",
    "train"
]

DEVICE = 0 if torch.cuda.is_available() else "cpu"
WORKERS = 4
IMAGE_SIZE = 1024
BATCH_SIZE = 16
EPOCHS = 20
MODEL_PATH = "yolo26n-cls.pt"


In [ ]:
def check_fold(fold_path: Path) -> None:
    """Check whether one fold has the expected directory structure."""

    train_path = fold_path / "train"
    val_path = fold_path / "val"

    if not train_path.exists():
        raise FileNotFoundError(
            f"Missing training directory: {train_path}"
        )

    if not val_path.exists():
        raise FileNotFoundError(
            f"Missing validation directory: {val_path}"
        )

    train_classes = sorted(
        path.name
        for path in train_path.iterdir()
        if path.is_dir()
    )

    val_classes = sorted(
        path.name
        for path in val_path.iterdir()
        if path.is_dir()
    )

    if not train_classes:
        raise ValueError(
            f"No class folders found in {train_path}"
        )

    if train_classes != val_classes:
        raise ValueError(
            f"Class folders do not match in {fold_path.name}.\n"
            f"Train classes: {train_classes}\n"
            f"Val classes:   {val_classes}"
        )

    print(f"Classes: {train_classes}")

In [ ]:
from pathlib import Path
import torchvision.transforms as T
from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import (
    ClassificationTrainer,
    ClassificationValidator,
)






class NoCropClassificationDataset(ClassificationDataset):
    """Classification dataset that resizes without cropping."""

    def __init__(
        self,
        root: str,
        args,
        augment: bool = False,
        prefix: str = "",
    ):
        super().__init__(root, args, augment, prefix)

        if augment:
            self.torch_transforms = T.Compose(
                [
                    # Preserve the complete 1024 × 1024 image.
                    T.Resize(
                        (args.imgsz, args.imgsz),
                        antialias=True,
                    ),

                    # Safe augmentations that do not crop the image.
                    T.RandomHorizontalFlip(p=0.5),

                    # Remove this if image orientation is meaningful.
                    T.RandomVerticalFlip(p=0.5),

                    # Current setting
                    T.ColorJitter(
                        brightness=0.1,
                        contrast=0.1,
                        saturation=0.1,
                        hue=0.02,
                    ),

                    T.ToTensor(),
                    T.Normalize(
                        mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225],
                    ),
                ]
            )
        else:
            self.torch_transforms = T.Compose(
                [
                    T.Resize(
                        (args.imgsz, args.imgsz),
                        antialias=True,
                    ),
                    T.ToTensor(),
                    T.Normalize(
                        mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225],
                    ),
                ]
            )


class NoCropClassificationTrainer(ClassificationTrainer):
    """Classification trainer using the no-crop dataset."""

    def build_dataset(
        self,
        img_path: str,
        mode: str = "train",
        batch=None,
    ):
        return NoCropClassificationDataset(
            root=img_path,
            args=self.args,
            augment=mode == "train",
            prefix=mode,
        )


class NoCropClassificationValidator(ClassificationValidator):
    """Validator using the same no-crop preprocessing."""

    def build_dataset(
        self,
        img_path: str,
        mode: str = "val",
    ):
        return NoCropClassificationDataset(
            root=img_path,
            args=self.args,
            augment=False,
            prefix=mode,
        )




Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
def train_fold(fold_name: str) -> None:
    fold_path = DATASET_ROOT / fold_name
    save_path = RESULT_ROOT / fold_name

    print("\n" + "=" * 60)
    print(f"Training: {fold_name}")
    print(f"Dataset:  {fold_path}")
    print(f"Results:  {save_path}")
    print("=" * 60)

    check_fold(fold_path)

    # Every fold starts from the same pretrained weights.
    model = YOLO(MODEL_PATH)

    model.train(
        data=str(fold_path),
        trainer=NoCropClassificationTrainer,
        task="classify",
        imgsz=IMAGE_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,

        save_dir=str(save_path),
        exist_ok=True,

        pretrained=True,
        optimizer="auto",
        seed=42,
        deterministic=True,
        patience=20,

        save=True,
        save_period=10,
        plots=True,
        verbose=True,
    )

    best_model_path = save_path / "weights" / "best.pt"

    if not best_model_path.exists():
        raise FileNotFoundError(
            f"Best model not found: {best_model_path}"
        )

    best_model = YOLO(str(best_model_path))

    metrics = best_model.val(
        data=str(fold_path),
        validator=NoCropClassificationValidator,
        split="val",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=WORKERS,
        save_dir=str(save_path / "validation"),
        plots=True,
    )

    print(f"\n{fold_name} results")
    print(f"Top-1 accuracy: {metrics.top1:.4f}")
    print(f"Top-5 accuracy: {metrics.top5:.4f}")

In [ ]:
# test: no crop 512
# test: 1024 run (is the features important)


def main() -> None:
    RESULT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"Using device: {DEVICE}")

    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    completed_folds = []
    failed_folds = []

    for fold_name in FOLDS:
        print(f"starting to work on train fold {fold_name}")
        try:
            train_fold(fold_name)
            completed_folds.append(fold_name)

        except Exception as error:
            print(f"\nFailed to train {fold_name}:")
            print(error)
            failed_folds.append(fold_name)

    print("\n" + "=" * 60)
    print("Cross-validation training finished")
    print(f"Completed folds: {completed_folds}")
    print(f"Failed folds:    {failed_folds}")
    print("=" * 60)


if __name__ == "__main__":
    main()


### Analysis for Individual Images

By processing individual validation images at a time. there will be more information outputted, like information the Stage 1 - 9 convolution features and the final label for the specific image.

Use concurrent library to speed up the process.

Note: This process can take up lots of space on Drive.


In [ ]:
from pathlib import Path
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
import threading


DATASET_PATH = Path(r"/content/drive/MyDrive/AI_bile_training")
round_name = "train"
MODEL_PATH = DATASET_PATH / "result" / round_name / "weights" / "best.pt"

# model = YOLO(f"{DATASET_PATH}/result/{round_name}/weights/best.pt")
val_dir = Path(f"{DATASET_PATH}/{round_name}/val")

output_root = Path(f"{DATASET_PATH}/result/activation_results")
output_root.mkdir(parents=True, exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

image_paths = [
    path
    for path in val_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in image_extensions
]


selected_paths = image_paths # one can slice the image_path as needed

thread_local = threading.local()


def get_model():
    if not hasattr(thread_local, "model"):
        print("Creating model for this worker thread")
        thread_local.model = YOLO(str(MODEL_PATH))

    return thread_local.model


def process_image(image_path: Path) -> dict:
    # the model won't be creating repeatedly
    local_model = get_model()

    true_class = image_path.parent.name

    results = local_model.predict(
        source=str(image_path),
        visualize=True,
        save=True,
        project=str(output_root / true_class),
        name=image_path.stem,
        exist_ok=True,
        verbose=False,
    )

    result = results[0]

    predicted_id = int(result.probs.top1)
    predicted_class = result.names[predicted_id]
    confidence = float(result.probs.top1conf)

    return {
        "image": image_path.name,
        "true_class": true_class,
        "predicted_class": predicted_class,
        "confidence": confidence,
        "correct": predicted_class == true_class,
    }

records = []
with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_path = {
        executor.submit(process_image, image_path): image_path
        for image_path in selected_paths
    }

    for future in as_completed(future_to_path):
        image_path = future_to_path[future]

        try:
            record = future.result()
            records.append(record)

            print(f"\nImage: {record['image']}")
            print(f"True: {record['true_class']}")
            print(f"Predicted: {record['predicted_class']}")
            print(f"Confidence: {record['confidence']:.4f}")
            print(f"Correct: {record['correct']}")

        except Exception as error:
            print(f"Failed on {image_path.name}: {error}")


# Write the CSV only after all threads have finished.
csv_path = output_root / "prediction_results.csv"

fieldnames = [
    "image",
    "true_class",
    "predicted_class",
    "confidence",
    "correct",
]

records.sort(key=lambda record: record["image"])

with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

print(f"\nCSV saved to: {csv_path}")
